<a href="https://colab.research.google.com/github/vaishnusuryawanshi27-tech/nassau-candy-route-efficiency-analysis/blob/main/Nassu_candy_cleaning_%26_streamlit_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Nassau Candy Distributor.csv')

# Display the first few rows and information about the date columns
print("Initial DataFrame Head:")
display(df[['Order Date', 'Ship Date']].head())

print("\nInitial Data Types:")
display(df[['Order Date', 'Ship Date']].info())

Initial DataFrame Head:


,Order Date,Ship Date
0,03-01-2024,30-06-2026
1,04-01-2024,01-07-2026
2,04-01-2024,01-07-2026
3,04-01-2024,01-07-2026
4,05-01-2024,05-07-2026



Initial Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Order Date  10194 non-null  object
 1   Ship Date   10194 non-null  object
dtypes: object(2)
memory usage: 159.4+ KB


None

In [2]:
# Convert 'Order Date' and 'Ship Date' to datetime objects
# Using dayfirst=True to correctly parse DD-MM-YYYY format

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')

# Identify rows where date conversion failed (if any)
invalid_order_dates = df[df['Order Date'].isna()]
invalid_ship_dates = df[df['Ship Date'].isna()]

print("Number of rows with invalid Order Dates after conversion:", len(invalid_order_dates))
print("Number of rows with invalid Ship Dates after conversion:", len(invalid_ship_dates))

# Display min and max dates to confirm the offset observation
print("\nMin Order Date:", df['Order Date'].min())
print("Max Order Date:", df['Order Date'].max())
print("Min Ship Date:", df['Ship Date'].min())
print("Max Ship Date:", df['Ship Date'].max())

Number of rows with invalid Order Dates after conversion: 0
Number of rows with invalid Ship Dates after conversion: 0

Min Order Date: 2024-01-02 00:00:00
Max Order Date: 2025-12-31 00:00:00
Min Ship Date: 2026-06-30 00:00:00
Max Ship Date: 2030-06-28 00:00:00


In [3]:
# Correct Ship Date by subtracting 2 years
df['Ship Date Corrected'] = df['Ship Date'] - pd.DateOffset(years=2)

# Calculate Shipping Lead Time
df['Shipping Lead Time'] = (
    df['Ship Date Corrected'] - df['Order Date']
).dt.days

# Check statistics
print(df['Shipping Lead Time'].describe())

# Check negative lead times
negative_lead_times = df[df['Shipping Lead Time'] < 0]

print("Negative Lead Times:", len(negative_lead_times))

# Preview
display(
    df[['Order Date',
        'Ship Date',
        'Ship Date Corrected',
        'Shipping Lead Time']].head()
)

count    10194.000000
mean       590.307043
std        262.307839
min        174.000000
25%        540.000000
50%        543.000000
75%        907.000000
max        912.000000
Name: Shipping Lead Time, dtype: float64
Negative Lead Times: 0


,Order Date,Ship Date,Ship Date Corrected,Shipping Lead Time
0,2024-01-03,2026-06-30,2024-06-30,179
1,2024-01-04,2026-07-01,2024-07-01,179
2,2024-01-04,2026-07-01,2024-07-01,179
3,2024-01-04,2026-07-01,2024-07-01,179
4,2024-01-05,2026-07-05,2024-07-05,182


In [4]:
# Define realistic lead time range
MIN_REALISTIC_LEAD_TIME = 1
MAX_REALISTIC_LEAD_TIME = 30

# Filter out invalid shipment records based on the realistic lead time range
df_cleaned = df[(df['Shipping Lead Time'] >= MIN_REALISTIC_LEAD_TIME) &
                (df['Shipping Lead Time'] <= MAX_REALISTIC_LEAD_TIME)].copy()

# Report on the number of records removed
records_removed = len(df) - len(df_cleaned)
print(f"Number of records before filtering: {len(df)}")
print(f"Number of records after filtering for realistic lead times ({MIN_REALISTIC_LEAD_TIME}-{MAX_REALISTIC_LEAD_TIME} days): {len(df_cleaned)}")
print(f"Number of records removed: {records_removed}")

# Display descriptive statistics for the cleaned Shipping Lead Time
print("\nDescriptive statistics for Cleaned Shipping Lead Time (days):")
display(df_cleaned['Shipping Lead Time'].describe())

# Display a few rows of the cleaned data to show the effect
print("\nCleaned DataFrame head with realistic Lead Times:")
display(df_cleaned[['Order Date', 'Ship Date', 'Ship Date Corrected', 'Shipping Lead Time']].head())

Number of records before filtering: 10194
Number of records after filtering for realistic lead times (1-30 days): 0
Number of records removed: 10194

Descriptive statistics for Cleaned Shipping Lead Time (days):


,Shipping Lead Time
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN



Cleaned DataFrame head with realistic Lead Times:


,Order Date,Ship Date,Ship Date Corrected,Shipping Lead Time


### Professional Cleaning Strategy: Categorizing Shipping Lead Time

Given that a strict 1-30 day `Shipping Lead Time` filter would remove the majority of the data, we'll adopt a strategy that preserves the dataset while still allowing for a meaningful analysis of shipping efficiency. The core idea is to categorize the *existing* `Shipping Lead Time` (after the 2-year correction) into professional groups such as 'Fast', 'Moderate', and 'Slow'. This allows for relative comparison and identification of bottlenecks within the dataset's inherent operational context.

**Strategy:**
1.  **Keep the `df` dataframe with the `Shipping Lead Time` (corrected for 2 years).** This column, though still high in absolute terms, consistently reflects the observed lead times after addressing the primary systemic error.
2.  **Define categories based on `Shipping Lead Time` distribution.** We will use percentiles (e.g., 33rd, 66th) to divide the lead times into 'Fast', 'Moderate', and 'Slow' groups. This is a data-driven approach that adapts to the unique characteristics of your dataset.
3.  **Create a new categorical column** for analysis.
4.  **Validate the distribution** of these categories.

In [5]:
# Calculate percentiles for Shipping Lead Time to define categories
# Using the dataframe 'df' that contains all records with the corrected lead time
lead_time_33rd_percentile = df['Shipping Lead Time'].quantile(0.33)
lead_time_66th_percentile = df['Shipping Lead Time'].quantile(0.66)

print(f"33rd percentile of Shipping Lead Time: {lead_time_33rd_percentile:.0f} days")
print(f"66th percentile of Shipping Lead Time: {lead_time_66th_percentile:.0f} days")

# Define a function to categorize lead times
def categorize_lead_time(lead_time):
    if lead_time <= lead_time_33rd_percentile:
        return 'Fast'
    elif lead_time <= lead_time_66th_percentile:
        return 'Moderate'
    else:
        return 'Slow'

# Apply the categorization to create a new column
df['Lead Time Category'] = df['Shipping Lead Time'].apply(categorize_lead_time)

# Display the distribution of the new categories
print("\nDistribution of Lead Time Categories:")
display(df['Lead Time Category'].value_counts())

# Display a few rows with the new category
print("\nDataFrame head with Lead Time Category:")
display(df[['Order Date', 'Ship Date Corrected', 'Shipping Lead Time', 'Lead Time Category']].head())

33rd percentile of Shipping Lead Time: 541 days
66th percentile of Shipping Lead Time: 545 days

Distribution of Lead Time Categories:


,count
Lead Time Category,
Slow,3458
Fast,3428
Moderate,3308



DataFrame head with Lead Time Category:


,Order Date,Ship Date Corrected,Shipping Lead Time,Lead Time Category
0,2024-01-03,2024-06-30,179,Fast
1,2024-01-04,2024-07-01,179,Fast
2,2024-01-04,2024-07-01,179,Fast
3,2024-01-04,2024-07-01,179,Fast
4,2024-01-05,2024-07-05,182,Fast


### Feature Engineering: Creating New Analytical Columns

Now that the date columns are cleaned and lead times are categorized, we will create additional features to support the various analytical requirements for your logistics project. These features will enable deeper insights into route efficiency, bottlenecks, and overall performance.

In [6]:
# 1. Create Factory Column
# Define the Product-to-Factory mapping
product_to_factory_mapping = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts", # Corrected: Key now precisely matches 'Wonka Bar -Scrumdiddlyumptious'
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Fizzy Lifting Drinks": "Sugar Shack",
    "Everlasting Gobstopper": "Secret Factory",
    "Hair Toffee": "The Other Factory",
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Kazookles": "The Other Factory"
}

# Standardize Product Name formatting before mapping
df['Product Name'] = df['Product Name'].str.strip() # Trim leading/trailing spaces
df['Product Name'] = df['Product Name'].str.replace(r'\s+', ' ', regex=True) # Replace multiple spaces with single space

# Apply the mapping to create the 'Factory' column
df['Factory'] = df['Product Name'].map(product_to_factory_mapping)

# Validate: Check for any products that did not get a factory assigned
missing_factories = df[df['Factory'].isna()]
if not missing_factories.empty:
    print("Warning: Some products do not have a mapped factory. Consider updating the mapping.")
    print("Products with missing factories:", missing_factories['Product Name'].unique())

print("Factory column created successfully.")
display(df[['Product Name', 'Factory']].head())

Factory column created successfully.


,Product Name,Factory
0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's


In [7]:
# 2. Create Factory-to-State Route Column
df['Factory-to-State Route'] = df['Factory'] + ' to ' + df['State/Province']

print("Factory-to-State Route column created successfully.")
display(df[['Factory', 'State/Province', 'Factory-to-State Route']].head())

Factory-to-State Route column created successfully.


,Factory,State/Province,Factory-to-State Route
0,Wicked Choccy's,Texas,Wicked Choccy's to Texas
1,Wicked Choccy's,Illinois,Wicked Choccy's to Illinois
2,Lot's O' Nuts,Illinois,Lot's O' Nuts to Illinois
3,Lot's O' Nuts,Illinois,Lot's O' Nuts to Illinois
4,Wicked Choccy's,Pennsylvania,Wicked Choccy's to Pennsylvania


In [8]:
# 3. Create Factory-to-Region Route Column
df['Factory-to-Region Route'] = df['Factory'] + ' to ' + df['Region']

print("Factory-to-Region Route column created successfully.")
display(df[['Factory', 'Region', 'Factory-to-Region Route']].head())

Factory-to-Region Route column created successfully.


,Factory,Region,Factory-to-Region Route
0,Wicked Choccy's,Interior,Wicked Choccy's to Interior
1,Wicked Choccy's,Interior,Wicked Choccy's to Interior
2,Lot's O' Nuts,Interior,Lot's O' Nuts to Interior
3,Lot's O' Nuts,Interior,Lot's O' Nuts to Interior
4,Wicked Choccy's,Atlantic,Wicked Choccy's to Atlantic


In [9]:
# 4. Create Delay Status Column
delay_status_mapping = {
    'Fast': 'On Time',
    'Moderate': 'Moderate Delay',
    'Slow': 'High Delay'
}

df['Delay Status'] = df['Lead Time Category'].map(delay_status_mapping)

print("Delay Status column created successfully.")
display(df[['Lead Time Category', 'Delay Status']].head())
display(df['Delay Status'].value_counts())

Delay Status column created successfully.


,Lead Time Category,Delay Status
0,Fast,On Time
1,Fast,On Time
2,Fast,On Time
3,Fast,On Time
4,Fast,On Time


,count
Delay Status,
High Delay,3458
On Time,3428
Moderate Delay,3308


In [10]:
# 5. Create Route Efficiency Score
# Lower lead time should mean higher efficiency. We can normalize this to a 0-100 scale.
# A common approach is to use the inverse or subtract from a max value and then scale.

# First, let's inverse the Lead Time so that smaller lead times yield larger values.
# To avoid division by zero or very large numbers with small lead times, we can add a small constant or use a scaled inverse.
# A simple approach is to calculate max_lead_time - actual_lead_time + 1
# Then, normalize this value to a 0-100 scale.

max_lead_time = df['Shipping Lead Time'].max()
min_lead_time = df['Shipping Lead Time'].min()

# Ensure there's a range to avoid division by zero if all lead times are identical
if max_lead_time == min_lead_time:
    df['Route Efficiency Score'] = 100 # All routes are equally efficient
else:
    # Calculate raw score where lower lead time means higher raw score
    df['Raw Efficiency Score'] = max_lead_time - df['Shipping Lead Time']
    # Normalize to a 0-100 scale. Max raw score corresponds to min lead time, min raw score to max lead time.
    df['Route Efficiency Score'] = ((df['Raw Efficiency Score'] - df['Raw Efficiency Score'].min()) /
                                    (df['Raw Efficiency Score'].max() - df['Raw Efficiency Score'].min())) * 100
    df = df.drop(columns=['Raw Efficiency Score'])

print("Route Efficiency Score column created successfully.")
display(df[['Shipping Lead Time', 'Route Efficiency Score']].head())
print("Descriptive statistics for Route Efficiency Score:")
display(df['Route Efficiency Score'].describe())

Route Efficiency Score column created successfully.


,Shipping Lead Time,Route Efficiency Score
0,179,99.322493
1,179,99.322493
2,179,99.322493
3,179,99.322493
4,182,98.915989


Descriptive statistics for Route Efficiency Score:


,Route Efficiency Score
count,10194.000000
mean,43.589832
std,35.543068
min,0.000000
25%,0.677507
50%,50.000000
75%,50.406504
max,100.000000


In [11]:
# 6. Create Delay Frequency Flag
# 'Delayed' for Moderate Delay and High Delay, 'Not Delayed' for On Time

def create_delay_flag(delay_status):
    if delay_status in ['Moderate Delay', 'High Delay']:
        return 'Delayed'
    else:
        return 'Not Delayed'

df['Delay Flag'] = df['Delay Status'].apply(create_delay_flag)

print("Delay Flag column created successfully.")
display(df[['Delay Status', 'Delay Flag']].head())
display(df['Delay Flag'].value_counts())

Delay Flag column created successfully.


,Delay Status,Delay Flag
0,On Time,Not Delayed
1,On Time,Not Delayed
2,On Time,Not Delayed
3,On Time,Not Delayed
4,On Time,Not Delayed


,count
Delay Flag,
Delayed,6766
Not Delayed,3428


### 7. Shipment Volume Readiness: Prepare for Route Aggregation Analysis

To analyze total shipments, average lead time, and lead time variability per route, we need to aggregate the dataset using the newly created route columns. This will provide key metrics for Power BI dashboarding and deeper analytical insights into route performance.

In [12]:
# Aggregate by Factory-to-State Route
route_state_agg = df.groupby('Factory-to-State Route').agg(
    Total_Shipments=('Order ID', 'count'),
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Lead_Time_Variability=('Shipping Lead Time', 'std')
).reset_index()

print("Aggregated by Factory-to-State Route:")
display(route_state_agg.head())

# Aggregate by Factory-to-Region Route
route_region_agg = df.groupby('Factory-to-Region Route').agg(
    Total_Shipments=('Order ID', 'count'),
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Lead_Time_Variability=('Shipping Lead Time', 'std')
).reset_index()

print("\nAggregated by Factory-to-Region Route:")
display(route_region_agg.head())

# Also consider aggregation by Ship Mode
ship_mode_agg = df.groupby('Ship Mode').agg(
    Total_Shipments=('Order ID', 'count'),
    Average_Lead_Time=('Shipping Lead Time', 'mean'),
    Lead_Time_Variability=('Shipping Lead Time', 'std')
).reset_index()

print("\nAggregated by Ship Mode:")
display(ship_mode_agg.head())

print("\nFinal DataFrame with all new feature engineering columns:")
display(df.head())

Aggregated by Factory-to-State Route:


,Factory-to-State Route,Total_Shipments,Average_Lead_Time,Lead_Time_Variability
0,Lot's O' Nuts to Alabama,34,564.705882,252.853476
1,Lot's O' Nuts to Alberta,16,543.375000,266.834749
2,Lot's O' Nuts to Arizona,111,575.684685,253.733145
3,Lot's O' Nuts to Arkansas,31,554.870968,274.464174
4,Lot's O' Nuts to British Columbia,18,543.333333,306.942991



Aggregated by Factory-to-Region Route:


,Factory-to-Region Route,Total_Shipments,Average_Lead_Time,Lead_Time_Variability
0,Lot's O' Nuts to Atlantic,1661,596.287778,255.875702
1,Lot's O' Nuts to Gulf,897,583.459309,266.448951
2,Lot's O' Nuts to Interior,1321,593.635882,258.268661
3,Lot's O' Nuts to Pacific,1813,586.987865,266.525868
4,Secret Factory to Atlantic,72,618.763889,280.650426



Aggregated by Ship Mode:


,Ship Mode,Total_Shipments,Average_Lead_Time,Lead_Time_Variability
0,First Class,1548,607.750646,265.504525
1,Same Day,547,602.884826,253.741691
2,Second Class,1979,593.300152,261.679332
3,Standard Class,6120,583.802778,262.253966



Final DataFrame with all new feature engineering columns:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Cost,Ship Date Corrected,Shipping Lead Time,Lead Time Category,Factory,Factory-to-State Route,Factory-to-Region Route,Delay Status,Route Efficiency Score,Delay Flag
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,2.28,2024-06-30,179,Fast,Wicked Choccy's,Wicked Choccy's to Texas,Wicked Choccy's to Interior,On Time,99.322493,Not Delayed
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,2.60,2024-07-01,179,Fast,Wicked Choccy's,Wicked Choccy's to Illinois,Wicked Choccy's to Interior,On Time,99.322493,Not Delayed
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.00,2024-07-01,179,Fast,Lot's O' Nuts,Lot's O' Nuts to Illinois,Lot's O' Nuts to Interior,On Time,99.322493,Not Delayed
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.30,2024-07-01,179,Fast,Lot's O' Nuts,Lot's O' Nuts to Illinois,Lot's O' Nuts to Interior,On Time,99.322493,Not Delayed
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,3.90,2024-07-05,182,Fast,Wicked Choccy's,Wicked Choccy's to Pennsylvania,Wicked Choccy's to Atlantic,On Time,98.915989,Not Delayed


### Professional Validation and Correction

#### Issue 1: Fix Encoding Issues in Route Columns

In [14]:
# Fix encoding issue in 'Factory-to-State Route'
df['Factory-to-State Route'] = df['Factory-to-State Route'].str.replace('â†’', ' to ', regex=False)

# Fix encoding issue in 'Factory-to-Region Route'
df['Factory-to-Region Route'] = df['Factory-to-Region Route'].str.replace('â†’', ' to ', regex=False)

print("Encoding issues in 'Factory-to-State Route' and 'Factory-to-Region Route' columns fixed.")

display(df[['Factory-to-State Route', 'Factory-to-Region Route']].head())

Encoding issues in 'Factory-to-State Route' and 'Factory-to-Region Route' columns fixed.


,Factory-to-State Route,Factory-to-Region Route
0,Wicked Choccy's to Texas,Wicked Choccy's to Interior
1,Wicked Choccy's to Illinois,Wicked Choccy's to Interior
2,Lot's O' Nuts to Illinois,Lot's O' Nuts to Interior
3,Lot's O' Nuts to Illinois,Lot's O' Nuts to Interior
4,Wicked Choccy's to Pennsylvania,Wicked Choccy's to Atlantic


#### Issue 2: Address Blank Values in Factory and Route Columns

In [15]:
# 1. Standardize Product Name formatting again to ensure consistency
# Trim leading/trailing spaces
df['Product Name'] = df['Product Name'].str.strip()
# Replace multiple spaces with single space
df['Product Name'] = df['Product Name'].str.replace(r'\s+', ' ', regex=True)

print("Product Name formatting standardized.")

# Correct the typo in the existing mapping for 'Wonka Bar -Scrumdiddlyumptious'
# and update the product_to_factory_mapping dictionary
if "Wonka Bar -Scrumdiddlyumptious" in product_to_factory_mapping:
    product_to_factory_mapping["Wonka Bar - Scrumdiddlyumptious"] = product_to_factory_mapping.pop("Wonka Bar -Scrumdiddlyumptious")


# 2. Rebuilding the Factory mapping
df['Factory'] = df['Product Name'].map(product_to_factory_mapping)

# Identify any remaining unmapped product names (where Factory is NaN)
unmapped_products = df[df['Factory'].isna()]['Product Name'].unique()

if len(unmapped_products) > 0:
    print(f"Warning: Found {len(unmapped_products)} unmapped product(s) after re-mapping:")
    for product in unmapped_products:
        print(f" - {product}")
        # Add unmapped products to the mapping, assigning to 'Unknown Factory'
        product_to_factory_mapping[product] = 'Unknown Factory'
    print("Unmapped products have been assigned to 'Unknown Factory' and mapping updated.")
    # Re-apply the mapping with the updated dictionary
    df['Factory'] = df['Product Name'].map(product_to_factory_mapping)
else:
    print("No unmapped products found after re-mapping.")

print("Factory column rebuilt successfully.")
display(df[['Product Name', 'Factory']].head())

Product Name formatting standardized.
 - Wonka Bar -Scrumdiddlyumptious
Unmapped products have been assigned to 'Unknown Factory' and mapping updated.
Factory column rebuilt successfully.


,Product Name,Factory
0,Wonka Bar - Milk Chocolate,Wicked Choccy's
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts
3,Wonka Bar -Scrumdiddlyumptious,Unknown Factory
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's


In [16]:
# 3. Recreate all route columns
# Factory-to-State Route
df['Factory-to-State Route'] = df['Factory'] + ' to ' + df['State/Province']

# Factory-to-Region Route
df['Factory-to-Region Route'] = df['Factory'] + ' to ' + df['Region']

print("Route columns 'Factory-to-State Route' and 'Factory-to-Region Route' recreated successfully.")

display(df[['Factory', 'State/Province', 'Factory-to-State Route']].head())
display(df[['Factory', 'Region', 'Factory-to-Region Route']].head())

Route columns 'Factory-to-State Route' and 'Factory-to-Region Route' recreated successfully.


,Factory,State/Province,Factory-to-State Route
0,Wicked Choccy's,Texas,Wicked Choccy's to Texas
1,Wicked Choccy's,Illinois,Wicked Choccy's to Illinois
2,Lot's O' Nuts,Illinois,Lot's O' Nuts to Illinois
3,Unknown Factory,Illinois,Unknown Factory to Illinois
4,Wicked Choccy's,Pennsylvania,Wicked Choccy's to Pennsylvania


,Factory,Region,Factory-to-Region Route
0,Wicked Choccy's,Interior,Wicked Choccy's to Interior
1,Wicked Choccy's,Interior,Wicked Choccy's to Interior
2,Lot's O' Nuts,Interior,Lot's O' Nuts to Interior
3,Unknown Factory,Interior,Unknown Factory to Interior
4,Wicked Choccy's,Atlantic,Wicked Choccy's to Atlantic


#### Validation: Check for Blank Values and Display Samples

In [17]:
# Display blank value counts for 'Factory', 'Factory-to-State Route', 'Factory-to-Region Route'
print("Blank value counts after correction:")
print(f"Factory: {df['Factory'].isnull().sum()}")
print(f"Factory-to-State Route: {df['Factory-to-State Route'].isnull().sum()}")
print(f"Factory-to-Region Route: {df['Factory-to-Region Route'].isnull().sum()}")

# Display 10 sample rows from these columns
print("\n10 sample rows from corrected columns:")
display(df[['Product Name', 'Factory', 'Factory-to-State Route', 'Factory-to-Region Route']].sample(10))

Blank value counts after correction:
Factory: 0
Factory-to-State Route: 0
Factory-to-Region Route: 0

10 sample rows from corrected columns:


,Product Name,Factory,Factory-to-State Route,Factory-to-Region Route
3729,Wonka Bar -Scrumdiddlyumptious,Unknown Factory,Unknown Factory to California,Unknown Factory to Pacific
7491,Wonka Bar - Milk Chocolate,Wicked Choccy's,Wicked Choccy's to California,Wicked Choccy's to Pacific
10187,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Lot's O' Nuts to Indiana,Lot's O' Nuts to Interior
7063,Wonka Bar -Scrumdiddlyumptious,Unknown Factory,Unknown Factory to Washington,Unknown Factory to Pacific
7041,Wonka Bar - Milk Chocolate,Wicked Choccy's,Wicked Choccy's to Ohio,Wicked Choccy's to Atlantic
6801,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Wicked Choccy's to Pennsylvania,Wicked Choccy's to Atlantic
202,Wonka Bar -Scrumdiddlyumptious,Unknown Factory,Unknown Factory to Mississippi,Unknown Factory to Gulf
9859,Kazookles,The Other Factory,The Other Factory to California,The Other Factory to Pacific
2782,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Wicked Choccy's to Pennsylvania,Wicked Choccy's to Atlantic
2370,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Lot's O' Nuts to California,Lot's O' Nuts to Pacific


#### Export Final Validated File

In [18]:
# Export a completely new final validated file
df.to_csv('nassau_candy_logistics_FINAL_VALIDATED_clean_dataset1.csv', index=False)

print("Final validated dataset exported successfully as 'nassau_candy_logistics_FINAL_VALIDATED.csv'!")

Final validated dataset exported successfully as 'nassau_candy_logistics_FINAL_VALIDATED.csv'!


In [19]:
!pip install streamlit pyngrok plotly --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.5 MB/s eta 0:00:00


In [20]:
import pandas as pd

# Load dataset
df = pd.read_csv('/content/nassau_candy_final_clean_dataset.csv')

# Check dataset
df.head()

,Row ID,Order ID,Order Date,Ship Date Corrected,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Gross Profit,Cost,Shipping Lead Time,Lead Time Category,Factory,Factory-to-State Route,Factory-to-Region Route,Delay Status,Route Efficiency Score,Delay Flag
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2024,Standard Class,103800,United States,Houston,Texas,77095,...,4.22,2.28,179,Fast,Wicked Choccy's,Wicked Choccy's to Texas,Wicked Choccy's to Interior,On Time,99.322493,Not Delayed
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2024,Standard Class,112326,United States,Naperville,Illinois,60540,...,4.90,2.60,179,Fast,Wicked Choccy's,Wicked Choccy's to Illinois,Wicked Choccy's to Interior,On Time,99.322493,Not Delayed
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2024,Standard Class,112326,United States,Naperville,Illinois,60540,...,7.47,3.00,179,Fast,Lot's O' Nuts,Lot's O' Nuts to Illinois,Lot's O' Nuts to Interior,On Time,99.322493,Not Delayed
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2024,Standard Class,112326,United States,Naperville,Illinois,60540,...,7.50,3.30,179,Fast,Unknown Factory,Unknown Factory to Illinois,Unknown Factory to Interior,On Time,99.322493,Not Delayed
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2024,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,7.35,3.90,182,Fast,Wicked Choccy's,Wicked Choccy's to Pennsylvania,Wicked Choccy's to Atlantic,On Time,98.915989,Not Delayed


In [33]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

# Page config
st.set_page_config(page_title="Nassau Candy Logistics Dashboard", layout="wide")

# Load dataset
df = pd.read_csv("/content/nassau_candy_final_clean_dataset.csv")

# Title
st.title("Nassau Candy Distributor Logistics Analysis")

# Sidebar filters
st.sidebar.header("Filters")

region = st.sidebar.multiselect(
    "Select Region",
    options=df["Region"].unique(),
    default=df["Region"].unique()
)

ship_mode = st.sidebar.multiselect(
    "Select Ship Mode",
    options=df["Ship Mode"].unique(),
    default=df["Ship Mode"].unique()
)

# Filter data
filtered_df = df[
    (df["Region"].isin(region)) &
    (df["Ship Mode"].isin(ship_mode))
]

# KPI Section
avg_lead = round(filtered_df["Shipping Lead Time"].mean(), 2)
total_orders = filtered_df["Order ID"].nunique()
avg_sales = round(filtered_df["Sales"].mean(), 2)
avg_profit = round(filtered_df["Gross Profit"].mean(), 2)

col1, col2, col3, col4 = st.columns(4)

col1.metric("Average Lead Time", avg_lead)
col2.metric("Total Orders", total_orders)
col3.metric("Average Sales", avg_sales)
col4.metric("Average Profit", avg_profit)

# Chart 1
st.subheader("Average Shipping Lead Time by Region")

region_chart = filtered_df.groupby("Region")["Shipping Lead Time"].mean().reset_index()

fig1 = px.bar(
    region_chart,
    x="Region",
    y="Shipping Lead Time",
    color="Shipping Lead Time"
)

st.plotly_chart(fig1, use_container_width=True)

# Chart 2
st.subheader("Ship Mode Performance")

ship_chart = filtered_df.groupby("Ship Mode")["Shipping Lead Time"].mean().reset_index()

fig2 = px.bar(
    ship_chart,
    x="Ship Mode",
    y="Shipping Lead Time",
    color="Shipping Lead Time"
)

st.plotly_chart(fig2, use_container_width=True)

# Chart 3
st.subheader("Delay Status Distribution")

fig3 = px.pie(
    filtered_df,
    names="Delay Status",
    hole=0.5
)

st.plotly_chart(fig3, use_container_width=True)

# Detailed Table
st.subheader("Detailed Shipment Data")

display_df = filtered_df[[
    "Order ID",
    "Order Date",
    "Ship Date Corrected",
    "Ship Mode",
    "Factory",
    "Region",
    "State/Province",
    "Shipping Lead Time",
    "Delay Status",
    "Sales",
    "Gross Profit"
]]

st.dataframe(display_df)

# Route Creation
filtered_df["Factory_to_State_Route"] = (
    filtered_df["Factory"] + " to " + filtered_df["State/Province"]
)

# Top 10 Efficient Routes
st.subheader("Top 10 Most Efficient Routes")

top_routes = (
    filtered_df.groupby("Factory_to_State_Route")["Shipping Lead Time"]
    .mean()
    .reset_index()
    .sort_values(by="Shipping Lead Time", ascending=True)
    .head(10)
)

fig4 = px.bar(
    top_routes,
    x="Shipping Lead Time",
    y="Factory_to_State_Route",
    orientation="h",
    color="Shipping Lead Time"
)

st.plotly_chart(fig4, use_container_width=True)

# Bottom 10 Least Efficient Routes
st.subheader("Bottom 10 Least Efficient Routes")

bottom_routes = (
    filtered_df.groupby("Factory_to_State_Route")["Shipping Lead Time"]
    .mean()
    .reset_index()
    .sort_values(by="Shipping Lead Time", ascending=False)
    .head(10)
)

fig5 = px.bar(
    bottom_routes,
    x="Shipping Lead Time",
    y="Factory_to_State_Route",
    orientation="h",
    color="Shipping Lead Time"
)

st.plotly_chart(fig5, use_container_width=True)

# ==============================
# LEAD TIME THRESHOLD SLIDER
# ==============================

st.subheader("Lead Time Threshold Filter")

lead_threshold = st.slider(
    "Select Minimum Shipping Lead Time",
    int(filtered_df["Shipping Lead Time"].min()),
    int(filtered_df["Shipping Lead Time"].max()),
    500
)

threshold_df = filtered_df[
    filtered_df["Shipping Lead Time"] >= lead_threshold
]

st.write("Filtered Records:", threshold_df.shape[0])

# ==============================
# STATE LEVEL BOTTLENECK ANALYSIS
# ==============================

st.subheader("Top Bottleneck States")

state_delay = (
    threshold_df.groupby("State/Province")["Shipping Lead Time"]
    .mean()
    .reset_index()
    .sort_values(by="Shipping Lead Time", ascending=False)
    .head(10)
)

fig6 = px.bar(
    state_delay,
    x="Shipping Lead Time",
    y="State/Province",
    orientation="h",
    color="Shipping Lead Time"
)

st.plotly_chart(fig6, use_container_width=True)

# ==============================
# REGION SHIPMENT VOLUME
# ==============================

st.subheader("Shipment Volume by Region")

region_volume = (
    filtered_df.groupby("Region")["Order ID"]
    .count()
    .reset_index()
)

fig7 = px.pie(
    region_volume,
    names="Region",
    values="Order ID"
)

st.plotly_chart(fig7, use_container_width=True)

Overwriting app.py


In [34]:
from pyngrok import ngrok

In [35]:
!streamlit run app.py &>/dev/null &

In [36]:
from pyngrok import ngrok

ngrok.set_auth_token("3ECcuzl0MftJsiTz0BbuQsuEATg_26UKjR1EZQEYGVGReeAVQ")

In [37]:
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://panama-mouth-generic.ngrok-free.dev" -> "http://localhost:8501">

In [26]:
from pyngrok import ngrok

ngrok.kill()

In [27]:
!streamlit run app.py &>/dev/null &

now save the dashboard app using this code

In [28]:
from google.colab import files

files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
files.download("nassau_candy_final_clean_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
%%writefile requirements.txt

streamlit
pandas
plotly
pyngrok

Writing requirements.txt
